# Calculate quantum yield

This notebook calculates $\Phi = n_\mathrm{reactive}/n_\mathrm{total}$. A valid trajectory is reactive only when its test-only cis/trans label changes between the first and final XYZ frame **and** its final MCH state is ground state 1. Final-excited trajectories are excluded from both counts.

(Ethene caveat: CH₂=CH₂ has no chemically meaningful cis/trans isomerism. The current results are a workflow test using a chosen H–C–C–H torsion.)

In [7]:
from pathlib import Path

# Run from either the repository root or sharc/.
SEARCH_ROOT = Path("sharc") if Path("sharc").is_dir() else Path(".")
TRAJ_GLOB = "structure_*/traj-allmols/**/TRAJ_*"

# 1-based H-C-C-H atom indices; change these for another molecule/order.
DIHEDRAL_ATOMS = (3, 1, 2, 4)
CIS_TRANS_THRESHOLD_DEG = 90.0
GROUND_MCH_STATE = 1

print(f"Searching: {SEARCH_ROOT.resolve()}")
print(f"Dihedral atoms: {DIHEDRAL_ATOMS}; ground MCH state: {GROUND_MCH_STATE}")

Searching: /home/lim_yt/X-MACE-sampling/sharc
Dihedral atoms: (3, 1, 2, 4); ground MCH state: 1


In [8]:
import math
import warnings

import numpy as np
import pandas as pd


def parse_xyz_lines(lines, source):
    """Return all conventional XYZ frames as (symbols, coordinates)."""
    frames, pos = [], 0
    while pos < len(lines):
        while pos < len(lines) and not lines[pos].strip():
            pos += 1
        if pos == len(lines):
            break
        try:
            natoms = int(lines[pos].strip())
        except ValueError as error:
            raise ValueError(f"{source}: expected atom count at line {pos + 1}") from error
        if natoms <= 0 or pos + natoms + 1 >= len(lines):
            raise ValueError(f"{source}: incomplete or invalid XYZ frame")
        symbols, xyz = [], []
        for line_number, line in enumerate(lines[pos + 2 : pos + 2 + natoms], pos + 3):
            fields = line.split()
            if len(fields) < 4:
                raise ValueError(f"{source}: malformed atom record at line {line_number}")
            try:
                xyz.append([float(value) for value in fields[1:4]])
            except ValueError as error:
                raise ValueError(f"{source}: non-numeric coordinate at line {line_number}") from error
            symbols.append(fields[0])
        frames.append((symbols, np.asarray(xyz, dtype=float)))
        pos += natoms + 2
    if not frames:
        raise ValueError(f"{source}: no XYZ frames")
    return frames


def signed_dihedral_deg(coordinates, atom_indices):
    indices = np.asarray(atom_indices, dtype=int) - 1
    if len(indices) != 4 or np.any(indices < 0) or np.any(indices >= len(coordinates)):
        raise ValueError(f"invalid atom indices {tuple(atom_indices)} for {len(coordinates)} atoms")
    p0, p1, p2, p3 = coordinates[indices]
    b0, b1, b2 = -(p1 - p0), p2 - p1, p3 - p2
    b1_norm = np.linalg.norm(b1)
    if b1_norm == 0:
        raise ValueError("middle dihedral bond has zero length")
    b1 /= b1_norm
    v, w = b0 - np.dot(b0, b1) * b1, b2 - np.dot(b2, b1) * b1
    if np.linalg.norm(v) == 0 or np.linalg.norm(w) == 0:
        raise ValueError("dihedral is undefined for collinear atoms")
    angle = math.degrees(math.atan2(np.dot(np.cross(b1, v), w), np.dot(v, w)))
    return ((angle + 180.0) % 360.0) - 180.0


def classify_dihedral(angle):
    return "cis" if abs(angle) <= CIS_TRANS_THRESHOLD_DEG else "trans"


def final_mch_state(lis_path):
    """Read the final SHARC output.lis data row: Step Time diag_state MCH_state ..."""
    rows = [line.split() for line in lis_path.read_text(encoding="utf-8").splitlines()
            if line.strip() and not line.lstrip().startswith("#")]
    if not rows or len(rows[-1]) < 4:
        raise ValueError(f"{lis_path}: no valid output.lis data row")
    try:
        return int(rows[-1][3])
    except ValueError as error:
        raise ValueError(f"{lis_path}: invalid final MCH state") from error


def outcome(initial, final, final_state):
    if final_state != GROUND_MCH_STATE:
        return "excluded_final_excited"
    return "reactive" if initial != final else "nonreactive"


def metadata(path):
    parts = path.parts
    pick = lambda prefix: next((part for part in parts if part.startswith(prefix)), "unknown")
    return pick("structure_"), pick("Singlet_"), pick("TRAJ_")

In [9]:
# In-notebook tests: parser, reactive directions, nonreactive, and excited-state exclusion.
xyz_fixture = """4
first
H 1 0 0
C 0 0 0
C 0 1 0
H 1 1 0
4
last
H 1 0 0
C 0 0 0
C 0 1 0
H -1 1 0
"""
frames = parse_xyz_lines(xyz_fixture.splitlines(), "fixture")
cis = classify_dihedral(signed_dihedral_deg(frames[0][1], (1, 2, 3, 4)))
trans = classify_dihedral(signed_dihedral_deg(frames[1][1], (1, 2, 3, 4)))
assert (cis, trans) == ("cis", "trans")
assert outcome(cis, trans, 1) == "reactive"
assert outcome(trans, cis, 1) == "reactive"
assert outcome(cis, cis, 1) == "nonreactive"
assert outcome(cis, trans, 2) == "excluded_final_excited"
print("Self-check passed.")

Self-check passed.


In [10]:
records = []
trajectory_dirs = sorted(path for path in SEARCH_ROOT.glob(TRAJ_GLOB) if path.is_dir())
if not trajectory_dirs:
    warnings.warn(f"No trajectory folders found under {SEARCH_ROOT}/{TRAJ_GLOB}")

for trajectory_dir in trajectory_dirs:
    xyz_path, lis_path = trajectory_dir / "output.xyz", trajectory_dir / "output.lis"
    structure, singlet, trajectory = metadata(trajectory_dir)
    record = {"structure": structure, "singlet": singlet, "trajectory": trajectory,
              "initial_dihedral_deg": np.nan, "initial_label": None,
              "final_dihedral_deg": np.nan, "final_label": None,
              "final_mch_state": pd.NA, "outcome": "skipped_invalid_output",
              "output_xyz": str(xyz_path), "reason": None}
    try:
        if not xyz_path.is_file() or xyz_path.stat().st_size == 0:
            raise ValueError("missing or empty output.xyz")
        if not lis_path.is_file() or lis_path.stat().st_size == 0:
            raise ValueError("missing or empty output.lis")
        frames = parse_xyz_lines(xyz_path.read_text(encoding="utf-8").splitlines(), str(xyz_path))
        initial_angle = signed_dihedral_deg(frames[0][1], DIHEDRAL_ATOMS)
        final_angle = signed_dihedral_deg(frames[-1][1], DIHEDRAL_ATOMS)
        final_state = final_mch_state(lis_path)
        record.update({"initial_dihedral_deg": initial_angle, "initial_label": classify_dihedral(initial_angle),
                       "final_dihedral_deg": final_angle, "final_label": classify_dihedral(final_angle),
                       "final_mch_state": final_state})
        record["outcome"] = outcome(record["initial_label"], record["final_label"], final_state)
    except (OSError, UnicodeDecodeError, ValueError) as error:
        record["reason"] = str(error)
        warnings.warn(f"Skipped {trajectory_dir}: {error}")
    records.append(record)

results = pd.DataFrame(records)
for column in ("initial_dihedral_deg", "final_dihedral_deg"):
    if column in results:
        results[column] = results[column].round(3)
results

,structure,singlet,trajectory,initial_dihedral_deg,initial_label,final_dihedral_deg,final_label,final_mch_state,outcome,output_xyz,reason
0,structure_0001,Singlet_1,TRAJ_00001,-2.589,cis,72.423,cis,2,excluded_final_excited,structure_0001/traj-allmols/Singlet_1/TRAJ_000...,None
1,structure_0002,Singlet_1,TRAJ_00001,-2.560,cis,72.801,cis,2,excluded_final_excited,structure_0002/traj-allmols/Singlet_1/TRAJ_000...,None
2,structure_0003,Singlet_1,TRAJ_00001,-2.585,cis,70.816,cis,2,excluded_final_excited,structure_0003/traj-allmols/Singlet_1/TRAJ_000...,None


In [11]:
eligible = results[results["outcome"].isin(["reactive", "nonreactive"])]
n_reactive = int((results["outcome"] == "reactive").sum())
n_total = len(eligible)  # valid trajectories ending in the ground MCH state
n_excluded_excited = int((results["outcome"] == "excluded_final_excited").sum())
n_nonreactive = int((results["outcome"] == "nonreactive").sum())
quantum_yield = n_reactive / n_total if n_total else np.nan

summary = pd.DataFrame([{
    "n_reactive": n_reactive,
    "n_total_ground_state": n_total,
    "n_nonreactive": n_nonreactive,
    "n_excluded_final_excited": n_excluded_excited,
    "n_skipped_invalid_output": int((results["outcome"] == "skipped_invalid_output").sum()),
    "quantum_yield": quantum_yield,
}])
display(summary)
if n_total == 0:
    print("Quantum yield is undefined since no trajectory ended in MCH ground state 1.")
else:
    print(f"Quantum yield = {quantum_yield:.4f} ({n_reactive}/{n_total})")

,n_reactive,n_total_ground_state,n_nonreactive,n_excluded_final_excited,n_skipped_invalid_output,quantum_yield
0,0,0,0,3,0,NaN


Quantum yield is undefined since no trajectory ended in MCH ground state 1.
